# Content Publish Workflow

Fill in the form widgets below, click **Add to site**, then run the publish cell at the bottom.

- Project / Demo / Poster / Publication / News → appended to `content/entries.json`
- Blog post → written as a markdown file under `content/blog/`

If widgets don't render, run `pip install ipywidgets` and reload the notebook.

In [1]:
from pathlib import Path
import json, re, subprocess, sys
import ipywidgets as widgets
from IPython.display import display, clear_output

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / 'scripts' / 'build_content.py').exists():
            return current
        current = current.parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
CONTENT_DIR = REPO_ROOT / 'content'
BLOG_DIR = CONTENT_DIR / 'blog'
ENTRIES_PATH = CONTENT_DIR / 'entries.json'
print('Repo root:', REPO_ROOT)

Repo root: /Users/rosalyn/GitHub/hyunjoors.github.io


## 1. Pick a content type

In [6]:
TYPE_CHOICES = ['project', 'demo', 'poster', 'publication', 'news', 'blog']

type_picker = widgets.RadioButtons(
    options=TYPE_CHOICES,
    description='Type:',
    style={'description_width': 'initial'},
)

form_box = widgets.VBox()
status_out = widgets.Output()

def text(label, placeholder=''):
    return widgets.Text(description=label, placeholder=placeholder,
                        layout=widgets.Layout(width='600px'),
                        style={'description_width': '120px'})

def area(label, placeholder=''):
    return widgets.Textarea(description=label, placeholder=placeholder,
                            layout=widgets.Layout(width='600px', height='120px'),
                            style={'description_width': '120px'})

def dropdown(label, options, value=None):
    return widgets.Dropdown(description=label, options=options,
                            value=value if value is not None else options[0],
                            layout=widgets.Layout(width='600px'),
                            style={'description_width': '120px'})

def date(label):
    return widgets.DatePicker(description=label,
                              layout=widgets.Layout(width='600px'),
                              style={'description_width': '120px'})

def build_form(kind):
    common = {
        'id': text('id', f'{kind}-my-slug'),
        'title': text('title', 'Title shown on cards'),
        'date': date('date'),
        'summary': area('summary', 'Short summary used on cards'),
        'tags': text('tags', 'comma,separated,tags'),
        'status': text('status', 'Live / Pilot / Presented / Announced ...'),
        'action_label': text('action label', 'Launch / Visit / View ...'),
        'action_url': text('action url', 'projects/.../index.html or https://...'),
        'action_target': dropdown('action target', ['embed', 'new_tab', 'same_tab'], 'new_tab'),
    }

    if kind == 'project':
        extras = {
            'mode': dropdown('mode', ['embedded', 'external'], 'external'),
            'role': text('role', 'Lead Developer & Designer'),
            'tagline': text('tagline', 'One-line positioning'),
        }
    elif kind == 'demo':
        extras = {
            'mode': dropdown('mode', ['embedded', 'external'], 'embedded'),
            'role': text('role', 'Lead Developer'),
            'venue': text('venue', 'Class / Lab / Conference'),
        }
    elif kind == 'poster':
        extras = {
            'venue': text('venue', 'LAK 2026, etc.'),
            'authors': text('authors', 'A, B, C'),
        }
    elif kind == 'publication':
        extras = {
            'venue': text('venue', 'Journal / Conference'),
            'authors': text('authors', 'A, B, C'),
        }
    elif kind == 'news':
        extras = {
            'newsType': dropdown('newsType', ['Milestone', 'Award', 'Talk', 'Press', 'Update'], 'Milestone'),
        }
    elif kind == 'blog':
        return {
            'id': text('id', 'my-post-slug'),
            'title': text('title', 'Post title'),
            'date': date('date'),
            'tag': text('tag', 'Research / Reflection / ...'),
            'excerpt': area('excerpt', 'One-sentence teaser'),
            'readTime': text('readTime', '6 min'),
            'body': area('body (markdown)', 'Write the post in markdown...'),
        }

    return {**common, **extras}

fields = {}

def render_form(_=None):
    global fields
    fields = build_form(type_picker.value)
    if type_picker.value == 'blog':
        fields['body'].layout = widgets.Layout(width='600px', height='300px')
    form_box.children = list(fields.values())

type_picker.observe(render_form, names='value')
render_form()

display(type_picker, form_box)

RadioButtons(description='Type:', options=('project', 'demo', 'poster', 'publication', 'news', 'blog'), style=…

## 2. Preview + add to site

In [7]:
preview_btn = widgets.Button(description='Preview', button_style='info')
save_btn = widgets.Button(description='Add to site', button_style='success')
overwrite = widgets.Checkbox(value=False, description='Overwrite if id exists')
out = widgets.Output()

def split_tags(s):
    return [t.strip() for t in (s or '').split(',') if t.strip()]

def fmt_date(d):
    if d is None:
        raise ValueError('date is required')
    return d.isoformat()

def build_object():
    kind = type_picker.value
    f = {k: w.value for k, w in fields.items()}

    if kind == 'blog':
        for k in ('id', 'title', 'tag', 'excerpt', 'readTime', 'body'):
            if not f.get(k):
                raise ValueError(f'{k} is required')
        return {
            'id': f['id'].strip(),
            'title': f['title'].strip(),
            'date': fmt_date(f['date']),
            'tag': f['tag'].strip(),
            'excerpt': f['excerpt'].strip(),
            'readTime': f['readTime'].strip(),
            'body': f['body'].strip(),
        }

    for k in ('id', 'title', 'summary', 'status', 'action_label', 'action_url'):
        if not f.get(k):
            raise ValueError(f'{k} is required')

    obj = {
        'id': f['id'].strip(),
        'type': kind,
        'title': f['title'].strip(),
        'date': fmt_date(f['date']),
        'summary': f['summary'].strip(),
        'tags': split_tags(f['tags']),
        'status': f['status'].strip(),
        'action': {
            'label': f['action_label'].strip(),
            'url': f['action_url'].strip(),
            'target': f['action_target'],
        },
    }
    for k in ('mode', 'role', 'tagline', 'venue', 'authors', 'newsType'):
        if k in f and f[k]:
            obj[k] = f[k].strip() if isinstance(f[k], str) else f[k]
    return obj

def on_preview(_):
    with out:
        clear_output()
        try:
            obj = build_object()
            print(json.dumps(obj, ensure_ascii=False, indent=2))
        except Exception as e:
            print('Error:', e)

def save_entry(obj):
    data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
    entries = data['entries']
    idx = next((i for i, e in enumerate(entries) if e['id'] == obj['id']), None)
    if idx is not None:
        if not overwrite.value:
            raise ValueError(f"id '{obj['id']}' already exists. Tick 'Overwrite' to replace.")
        entries[idx] = obj
    else:
        entries.append(obj)
    ENTRIES_PATH.write_text(json.dumps({'entries': entries}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return 'updated' if idx is not None else 'appended'

def save_blog(obj):
    slug = re.sub(r'[^a-zA-Z0-9-]+', '-', obj['id']).strip('-').lower()
    path = BLOG_DIR / f'{slug}.md'
    if path.exists() and not overwrite.value:
        raise ValueError(f"{path.name} already exists. Tick 'Overwrite' to replace.")
    frontmatter = (
        '---\n'
        f"id: {obj['id']}\n"
        f"title: {obj['title']}\n"
        f"date: {obj['date']}\n"
        f"tag: {obj['tag']}\n"
        f"excerpt: {obj['excerpt']}\n"
        f"readTime: {obj['readTime']}\n"
        '---\n\n'
        f"{obj['body']}\n"
    )
    path.write_text(frontmatter, encoding='utf-8')
    return path.relative_to(REPO_ROOT)

def on_save(_):
    with out:
        clear_output()
        try:
            obj = build_object()
            if type_picker.value == 'blog':
                path = save_blog(obj)
                print('Wrote', path)
            else:
                action = save_entry(obj)
                print(f"{action} entry '{obj['id']}' in content/entries.json")
            print('Next: run the publish cell below.')
        except Exception as e:
            print('Error:', e)

preview_btn.on_click(on_preview)
save_btn.on_click(on_save)
display(widgets.HBox([preview_btn, save_btn, overwrite]), out)

Output()

## 3. Build

Regenerates `site/generated/` from `content/`. Run this after adding entries so the site picks up the changes locally.

In [4]:
build_btn = widgets.Button(description='Build', button_style='info')
build_out = widgets.Output()
build_ok = {'value': False}

def on_build(_):
    with build_out:
        clear_output()
        build = subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'build_content.py')],
                               cwd=REPO_ROOT, text=True, capture_output=True)
        print(build.stdout)
        if build.returncode != 0:
            print(build.stderr)
            print('Build failed')
            build_ok['value'] = False
            return
        build_ok['value'] = True
        print('Build OK.')

build_btn.on_click(on_build)
display(build_btn, build_out)

Button(button_style='info', description='Build', style=ButtonStyle())

Output()

## 4. Publish

Commits the changes (and optionally pushes to remote). Run **Build** first.

In [5]:
commit_msg = widgets.Text(description='commit msg', layout=widgets.Layout(width='600px'),
                          style={'description_width': '120px'})
branch = widgets.Text(description='branch', value='main',
                      layout=widgets.Layout(width='300px'),
                      style={'description_width': '120px'})
push = widgets.Checkbox(value=False, description='Push to remote')
skip_build_check = widgets.Checkbox(value=False, description='Skip build check')
publish_btn = widgets.Button(description='Publish', button_style='warning')
pub_out = widgets.Output()

def on_publish(_):
    with pub_out:
        clear_output()
        if not build_ok['value'] and not skip_build_check.value:
            print("Run Build first, or tick 'Skip build check'.")
            return
        cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'publish_content.py'),
               '--target-branch', branch.value]
        if commit_msg.value.strip():
            cmd += ['--commit-message', commit_msg.value.strip()]
        cmd.append('--push-confirm' if push.value else '--no-push')
        pub = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
        print(pub.stdout)
        if pub.stderr.strip():
            print(pub.stderr)

publish_btn.on_click(on_publish)
display(commit_msg, branch, push, skip_build_check, publish_btn, pub_out)

Text(value='', description='commit msg', layout=Layout(width='600px'), style=TextStyle(description_width='120p…

Text(value='main', description='branch', layout=Layout(width='300px'), style=TextStyle(description_width='120p…

Checkbox(value=False, description='Push to remote')

Checkbox(value=False, description='Skip build check')

Button(button_style='warning', description='Publish', style=ButtonStyle())

Output()